In [1]:
# !pip install tensorflow-addons==0.18.0

In [2]:
import tensorflow as tf
print("当前TensorFlow版本：", tf.__version__)

当前TensorFlow版本： 2.20.0


In [3]:
import sys
# 添加上级目录（models所在的"瑞金"文件夹路径）到系统搜索路径
sys.path.append(r"F:\AAAAAAAAAAAAAAAAAAAAA\Senior\subject\数据挖掘\实验十一\notebook-瑞金\瑞金")

import numpy as np  # 导入numpy，用于数值计算、矩阵构建与数据格式转换

from sklearn.model_selection import ShuffleSplit  # 导入随机划分工具，用于训练/测试集文档划分

# 从自定义工具包data_utils导入NER任务核心组件：
# ENTITIES：预定义的医疗实体类别（如疾病、症状、药品等）
# Documents：文档加载与封装类，用于读取原始医疗数据集
# Dataset：数据集封装类，用于文本数据数值化、词汇表构建
# SentenceExtractor：文本片段提取器，用于切分标准化文本窗口
# make_predictions：预测结果还原工具，将数值型预测转为可读实体格式
from data_utils import ENTITIES, Documents, Dataset, SentenceExtractor, make_predictions

from data_utils import Evaluator  # 从自定义工具包导入评估器，用于计算F1值、精确率、召回率

from models import build_lstm_crf_model  # 从自定义模型包导入LSTM-CRF模型构建函数

from gensim.models import Word2Vec  # 导入Word2Vec，用于预训练汉字语义向量

In [4]:
# !pip install https://github.com/keras-team/keras-contrib.git

In [5]:
# from keras_contrib.layers import CRF

In [6]:
# ************************ 数据路径与实体映射配置 ************************

data_dir = "F:/AAAAAAAAAAAAAAAAAAAAA/Senior/subject/数据挖掘/实验十一/notebook-瑞金/瑞金/ruijin_round1_train2_20181022/"  # 定义瑞金医院医疗数据集的存储根路径

ent2idx = dict(zip(ENTITIES, range(1, len(ENTITIES) + 1)))  # 构建「实体名称→整数索引」映射字典，索引从1开始（预留0为背景类）

idx2ent = dict([(v, k) for k, v in ent2idx.items()])  # 构建「整数索引→实体名称」反向映射字典，用于后续预测结果还原

In [7]:
# ************************ 数据加载与训练/测试集划分 ************************

docs = Documents(data_dir=data_dir)  # 实例化Documents类，加载指定路径下的所有医疗文档（包含文本与标注信息）

# 初始化ShuffleSplit随机划分器：
# n_splits=1：仅执行1次数据划分
# test_size=20：测试集包含20个文档
# random_state=2018：固定随机种子，保证每次运行划分结果一致，实现实验可复现
rs = ShuffleSplit(n_splits=1, test_size=20, random_state=2018)

train_doc_ids, test_doc_ids = next(rs.split(docs))  # 执行划分，获取训练集/测试集的文档ID列表

train_docs, test_docs = docs[train_doc_ids], docs[test_doc_ids]  # 根据文档ID，提取对应的训练文档集与测试文档集

In [8]:
train_docs[0]  # 调试/验证用：查看第一个训练文档的结构与内容，确认数据加载是否正常

In [9]:
# ************************ 预处理参数配置与文本片段提取 ************************

num_cates = max(ent2idx.values()) + 1  # 计算实体类别总数（含背景类），+1是为了预留索引0作为「非实体/背景类」

sent_len = 64  # 定义单个文本片段的核心窗口长度（有效文本长度）
vocab_size = 3000  # 预设词汇表大小（过滤低频汉字，控制模型参数量）
emb_size = 100  # 定义汉字向量的维度（预训练与模型嵌入层保持一致）
sent_pad = 10  # 定义文本片段前后的填充长度，避免边界信息丢失，保证样本长度统一

sent_extrator = SentenceExtractor(window_size=sent_len, pad_size=sent_pad)  # 实例化文本片段提取器，传入窗口大小与填充大小

train_sents = sent_extrator(train_docs)  # 对训练文档集进行文本切分与填充，生成标准化训练文本片段集合
test_sents = sent_extrator(test_docs)  # 对测试文档集进行文本切分与填充，生成标准化测试文本片段集合

In [10]:
# ************************ 数据集构建与数值化（核心预处理） ************************
train_data = Dataset(train_sents, cate2idx=ent2idx)  # 实例化Dataset类，封装训练集文本片段与实体标签映射
train_data.build_vocab_dict(vocab_size=vocab_size)  # 基于训练集文本构建「汉字→整数索引」映射字典，按预设词汇表大小过滤低频汉字

# 实例化Dataset类，封装测试集文本片段：
# 关键参数word2idx=train_data.word2idx：复用训练集的词汇表，避免数据泄露，保证模型泛化性
# 未知汉字将映射为统一的「未知字」索引
test_data = Dataset(test_sents, word2idx=train_data.word2idx, cate2idx=ent2idx)

vocab_size = len(train_data.word2idx)  # 更新词汇表大小为训练集词汇表的实际长度（可能≤预设的3000）

In [11]:
# ************************ Word2Vec字向量预训练与嵌入矩阵构建 ************************

w2v_train_sents = []  # 初始化Word2Vec预训练语料列表
for doc in docs:  # 遍历所有文档（而非仅训练集，提升字向量覆盖度）
    w2v_train_sents.append(list(doc.text))  # 将每个文档的文本转换为汉字列表，加入预训练语料

w2v_model = Word2Vec(w2v_train_sents, vector_size=emb_size)  # 实例化并训练Word2Vec模型，生成维度为emb_size的汉字语义向量

w2v_embeddings = np.zeros((vocab_size, emb_size))  # 初始化全0嵌入矩阵，形状为「词汇表大小×向量维度」
for char, char_idx in train_data.word2idx.items():  # 遍历训练集词汇表中的每个汉字与对应索引
    if char in w2v_model.wv:  # 若该汉字在预训练Word2Vec模型中有对应向量
        w2v_embeddings[char_idx] = w2v_model.wv[char]  # 将预训练向量填充到嵌入矩阵的对应索引位置

In [12]:
# 注：无对应预训练向量的汉字（低频/生僻字）保持全0，作为未知字向量

# ************************ LSTM-CRF模型构建 ************************

seq_len = sent_len + 2 * sent_pad  # 计算模型输入序列的总长度（核心窗口长度+前后填充长度）

# 实例化并构建LSTM-CRF模型：
# num_cates：实体类别总数（含背景类）
# seq_len：模型输入序列总长度
# vocab_size：训练集词汇表实际大小
# model_opts：模型配置参数字典：
# emb_matrix：预训练汉字嵌入矩阵，用于初始化嵌入层权重
# emb_size：嵌入层输出维度（与预训练向量维度一致）
# emb_trainable：冻结嵌入层权重（不参与模型训练），避免破坏预训练语义信息
model = build_lstm_crf_model(num_cates, static_seq_len=seq_len, vocab_size=vocab_size,
                             model_opts={'emb_matrix': w2v_embeddings, 'emb_size': 100, 'emb_trainable': False})

model.summary()  # 打印模型结构、各层参数量与总参数量，验证模型构建是否符合预期

模型构建成功，CRF层转移矩阵已初始化


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 84)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 84, 100)        │       300,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 84, 512)        │       731,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 84, 16)         │         8,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ custom_crf (CustomCRF)          │ (None, 84)             │           256 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,039,600 (3.97 MB)

 Trainable params: 739,600 (2.82 MB)

 Non-trainable params: 300,000 (1.14 MB)

In [13]:
# # ************************ 模型训练 ************************
#
# train_X, train_y = train_data[:]  # 提取训练集的数值化输入特征（train_X：汉字索引序列）与标签（train_y：实体索引序列）
# print('train_X.shape', train_X.shape)  # 打印训练集输入特征形状，验证是否适配模型输入要求（调试用）
# print('train_y.shape', train_y.shape)  # 打印训练集标签形状，验证与输入特征是否匹配（调试用）
# train_y = tf.squeeze(train_y, axis=-1)  # 挤压最后一个维度，从 (N, 84, 1) 转为 (N, 84)
#
# # 模型训练：
# # batch_size=64：每次训练批次大小为64个样本
# # epochs=10：训练总轮数为10轮
# model.fit(train_X, train_y, batch_size=64, epochs=10)

In [14]:
# ************************ 模型训练 ************************
train_X, train_y = train_data[:]  # 提取训练集

# 步骤1：验证并统一数据格式为 Numpy 数组
if isinstance(train_X, tf.Tensor):
    train_X = train_X.numpy()
if isinstance(train_y, tf.Tensor):
    train_y = train_y.numpy()

print('train_X.shape', train_X.shape)  # 预期 (47577, 84)
print('train_y.shape', train_y.shape)  # 预期 (47577, 84, 1)

# 步骤2：挤压最后一个维度（使用 Numpy 方法，避免生成 tf.Tensor）
train_y = train_y.squeeze(axis=-1)  # 改为 Numpy 的 squeeze，结果仍为 Numpy 数组
print('train_y after squeeze.shape', train_y.shape)  # 预期 (47577, 84)

# 步骤3：确保数据类型为 int32（匹配模型标签要求）
train_X = train_X.astype('int32')
train_y = train_y.astype('int32')

# 模型训练
model.fit(train_X, train_y, batch_size=64, epochs=10)

train_X.shape (47577, 84)
train_y.shape (47577, 84, 1)
train_y after squeeze.shape (47577, 84)
Epoch 1/10


ValueError: Can not squeeze dim[1], expected a dimension of 1, got 84 for '{{node compile_loss/loss/Squeeze}} = Squeeze[T=DT_FLOAT, squeeze_dims=[-1]](compile_loss/loss/Cast)' with input shapes: [?,84].

In [29]:
# ************************ 模型预测与预测结果还原 ************************

test_X, _ = test_data[:]  # 提取测试集的数值化输入特征，忽略测试集标签（标签仅用于后续评估，不参与预测）
preds = model.predict(test_X, batch_size=64, verbose=True)  # 对测试集进行批量预测，得到数值型实体索引序列预测结果

# 预测结果还原：将数值型preds转换为与原始文档格式一致的可读预测结果
# 依赖参数：测试集数据、填充长度、原始文档集、索引→实体映射字典
pred_docs = make_predictions(preds, test_data, sent_pad, docs, idx2ent)

AttributeError: Exception encountered when calling CRF.call().

[1mmodule 'keras.backend' has no attribute 'expand_dims'[0m

Arguments received by CRF.call():
  • X=tf.Tensor(shape=(64, 84, 512), dtype=float32)
  • mask=None

In [ ]:
# ************************ 模型性能评估 ************************

# 对比测试集真实标注（test_docs）与模型预测结果（pred_docs），计算核心评估指标

f_score, precision, recall = Evaluator.f1_score(test_docs, pred_docs)

print('f_score: ', f_score)  # 打印F1值（精确率与召回率的调和平均，综合反映模型性能）
print('precision: ', precision)  # 打印精确率（预测为实体的结果中，真实为实体的比例）
print('recall: ', recall)  # 打印召回率（真实为实体的内容中，被成功预测的比例）

In [ ]:
# ************************ 预测结果调试与人工验证 ************************

sample_doc_id = list(pred_docs.keys())[0]  # 获取预测结果中第一个文档的ID

test_docs[sample_doc_id]  # 查看该文档在测试集中的真实标注信息，用于人工对比真实标注与模型预测结果，排查偏差